In [14]:
from pathlib import Path

# Write the function
def read_file(file_path):
    """
    Reads a file using UTF-8 encoding, falling back to ISO-8859-1
    if UTF-8 decoding fails.

    Args:
        file_path: The path to the file to read.

    Returns:
        A tuple containing the file contents as a string and the
        encoding used ("UTF-8" or "ISO-8859-1").
    """

    file_path = Path(file_path)  # Convert the file path to a Path object

    try:
        text = file_path.read_text(encoding="utf-8") # reads the file with UTF-8
        return text, "UTF-8"                         # returns the text and encoding

    except UnicodeDecodeError:                            # if UTF-8 decoding fails, it falls back to ISO-8859-1
        text = file_path.read_text(encoding="ISO-8859-1") # reads the file with ISO-8859-1
        return text, "ISO-8859-1"                         # returns the text and encoding

    

In [13]:
import re 

# Write the function to parse the log file
def parse_log_file(file_text_content):
    """ 
    Parses the log file content and extracts transaction information.

    Args:
        file_text_content: The content of the log file as a string.

    Returns:
        A list of dictionaries, each containing information about a transaction.
    """

# Making the regex-pattern
    pattern = (    
        r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(POS_\d+)\] "
        r"item=(.*?) "
        r"price=(\d+\.\d{2}) "
        r"(EUR|NOK|USD) "
        r"\((SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions = []

    for line in file_text_content.splitlines():
        match = re.search(pattern, line)        # searches for matches in the line using the regex pattern

        if match:
            timestamp, register_id, item, price, currency, status = match.groups() # extracts the matched groups from the regex match

        # Create dictionary (for transaction)
            transaction = {
                "timestamp": timestamp,
                "register_id": register_id,
                "item": item,
                "price": float(price), 
                "currency": currency,
                "status": status
            }

            transactions.append(transaction)  # adds the new dictionary to the transactions list.

    return transactions # When all lines have been examined, --> function returns the list containing all the transactions it found.

# Task 2: Recursive utility functions
## Part 1: directory search
in this code we find all .log files within the data directory


In [15]:
from pathlib import Path

#function to find all .log files in a given folder and its subfolders, for the task this will be "data"
def find_log_files(folder_path):
    folder = Path(folder_path)
    log_files = []

    # recursive search for .log files
    for item in folder.iterdir():

        if item.is_dir():
            log_files.extend(find_log_files(item))  # recursive call for subdirectories

        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


#test the function by searching for .log files in the "data" folder
log_files = find_log_files("data")

print(f"Found {len(log_files)} log files:")

for log_file in log_files:
    print(log_file)


Found 0 log files:


## Part 2: Recursive search through JSON data

identifying active register IDs from the nested json file

In [ ]:
import json

with open("data/store_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

def find_active_registers(data) -> set[str]:
    active_registers = set()

    #check if the dictionairy represents an active register and search through its values recursively
    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        for value in data.values():
            active_registers.update(find_active_registers(value))

    #search through each element recursively
    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers 


#test the function by finding all active registers in the store_config.json file
active_registers = find_active_registers(config)
print("Active registers:", active_registers)
print("Number of active registers:", len(active_registers))



Active registers: {'POS_01', 'POS_02', 'POS_03'}
Number of active registers: 3
